***DEMO***

Demo to showcase the power of the Analyze endpoint and coreference features

Using the sample medical conversations below we use the analyze endpoint to parse out the coreference ids and extracted relations.
The coreference IDs show all of the different ways a single entity is referenced. 
Combined with relation extraction this shows the links between different entity types for DOB, origin, kinship and location.


2025-10-10: Dr. Samantha Lee saw patient Michael Carter at Toronto General Hospital in the Cardiology clinic. Mr. Carter is a 41 yo male with a 
known history of hypertension and asthma. During the visit, Dr. Lee prescribed LISINOPRIL and encouraged Micheal to monitor his BP daily at home and 
record the readings for his next visit.  Michael informed Dr. Lee of his upcoming trip. Michael's birthday is October 20th, and
he will be flying to Vancouver, BC this week to celebrate with family. Michael mentioned that his brother, David Carter, who was born
in Toronto, Ontario, now lives in Vancouver; recently recovered from knee surgery and will pick him up from the airport


2025-10-22: Dr. S. Lee met with Mr. Michael Carter at Toronto General Hospital in the Cardiology clinic. 
Mr. Carter is being followed for hypertension management.  Mike discussed his recent trip to British Columbia and provided and update on his
health. The patient indicated that the LISINOPRIL is working well.  His BP readings at home were WNL.  Mr. Carter expressed that he was 
thankful to be seen by Dr. Lee on such short notice. As a longtime resident of Toronto, Michael expressed his gratitude for having 
great medical professionals in the area.


2025-10-10: Dr. Jonathan Kim met with patient Ms. Emily Rodriguez at Mount Sinai Clinic. Emily is
a 24 year old female originally from Jamaica and has recently become a Canadian citizen. Ms. Rodriguez has a known history of managed by oral 
hypoglycemic medications.  No known drug allergies, but report seasonal allergies. Emily spoke about her brother, Alex Rodriguez, who 
is currently studying medicine and is going to the same school where Dr. Kim graduated from.

In [9]:

# This code assumes that you have the Private AI deidentification service running locally on port 8080 
# or you have a valid community api key saved in a .env file in the project folder that contains the variable
# PRIVATEAI_API_KEY = my_key 

import requests
import os
import dotenv
from collections import defaultdict

#Chart list for very chatty patients
# Uncomment and run each of these one at a time.
text = [
    """2025-10-10: Dr. Samantha Lee saw patient Michael Carter at Toronto General Hospital in the Cardiology clinic. Mr. Carter is a 41 yo male with a 
    known history of hypertension and asthma. During the visit, Dr. Lee prescribed LISINOPRIL and encouraged Micheal to monitor his BP daily at home and 
    record the readings for his next visit.  Michael informed Dr. Lee of his upcoming trip. Michael's birthday is October 20th, and
     he will be flying to Vancouver, BC this week to celebrate with family. Michael mentioned that his brother, David Carter, who was born
     in Toronto, Ontario, now lives in Vancouver; recently recovered from knee surgery and will pick him up from the airport """
     
    #  ,  

    # """2025-10-22: Dr. S. Lee met with Mr. Michael Carter at Toronto General Hospital in the Cardiology clinic. 
    # Mr. Carter is being followed for hypertension management.  Mike discussed his recent trip to British Columbia and provided and update on his
    # health. The patient indicated that the LISINOPRIL is working well.  His BP readings at home were WNL.  Mr. Carter expressed that he was 
    # thankful to be seen by Dr. Lee on such short notice. As a longtime resident of Toronto, Michael expressed his gratitude for having 
    # great medical professionals in the area."""

    # ,

    # """2025-10-10: Dr. Jonathan Kim met with patient Ms. Emily Rodriguez at Mount Sinai Clinic. Emily is
    #   a 24 year old female originally from Jamaica and has recently become a Canadian citizen. Ms. Rodriguez has a known history of managed by oral 
    #   hypoglycemic medications.  No known drug allergies, but report seasonal allergies. Emily spoke about her brother, Alex Rodriguez, who 
    #   is currently studying medicine and is going to the same school where Dr. Kim graduated from."""
]    
    
#Setup payload
request = {"text": text,
    "link_batch": False,
    "entity_detection": {
        "return_entity": True
    },
    "relation_detection": {
        "coreference_resolution" : "model_prediction",
        "enable_relation_extraction": True
    }  
}

###----------------------------------------------------------------------------###
### PRIVATE AI API CALL
dotenv.load_dotenv()
#url = "http://localhost:8080/analyze/text"
url = "https://api.private-ai.com/community/v4/analyze/text"

headers = {"Content-Type": "application/json", 
           "x-api-key":os.environ["PRIVATEAI_API_KEY"]}
response = requests.post(url, json=request, headers=headers)
data = response.json()

### PRIVATE AI API CALL
###----------------------------------------------------------------------------###

#Print pretty response to show the results
#The code below does JSON parsing to print data to screen
def print_entity_relations_grouped_by_coref_id(entities_array):
    """
    Groups entities by coreference_id and, for each group, prints the text, best_label, coreference_id,
    and for each relation, prints the related entity's text and best_label.
    """
    
    # Flatten all entities from the array
    all_entities = []
    for entry in entities_array:
        all_entities.extend(entry.get("entities", []))

    # Group entities by coreference_id
    coref_groups = defaultdict(list)
    for entity in all_entities:
        coref_id = entity.get("coreference_id")
        if coref_id is not None:
            coref_groups[coref_id].append(entity)

    # Build a lookup for coreference_id to entity
    entity_lookup = {e.get("coreference_id"): e for e in all_entities}

    # Print out each coreference group, the entities inside and the related entity ids
    for coref_id, group in coref_groups.items():        
        print(f"Coreference Group ID: {coref_id}")
        all_relations = []
        for entity in group:                    
            text = entity.get("text")
            best_label = entity.get("best_label")
            print(f"  Entity: '{text}' | Label: {best_label}")            
            all_relations.extend(entity.get("relations", []))

        if(all_relations):
            seen_coref_ids = set()
            for rel in all_relations:
                rel_id = rel.get("coreference_id")
                if rel_id not in seen_coref_ids:
                    seen_coref_ids.add(rel_id)
                    rel_entity = entity_lookup.get(rel_id)
                    if rel_entity:
                        rel_text = rel_entity.get("text")
                        rel_label = rel_entity.get("best_label")
                        print(f"    Related: '{rel_text}' | Label: {rel_label}")
        print("-" * 40)

#Run the coref demo
print_entity_relations_grouped_by_coref_id(data)                



Coreference Group ID: 2e329d72-4373-4bdf-8309-c1781d91eabd
  Entity: '2025-10-10' | Label: DATE
----------------------------------------
Coreference Group ID: 075eddfa-cd8e-4386-b572-48e8a2b10b11
  Entity: 'Dr.' | Label: OCCUPATION
----------------------------------------
Coreference Group ID: ba576732-e214-4c06-b942-293f2eb358de
  Entity: 'Samantha Lee' | Label: NAME_MEDICAL_PROFESSIONAL
  Entity: 'Lee' | Label: NAME_MEDICAL_PROFESSIONAL
  Entity: 'Lee' | Label: NAME_MEDICAL_PROFESSIONAL
----------------------------------------
Coreference Group ID: 624df560-41a4-443d-99c4-3c17684bdf04
  Entity: 'Michael Carter' | Label: NAME
  Entity: 'Carter' | Label: NAME_FAMILY
  Entity: 'Micheal' | Label: NAME_GIVEN
  Entity: 'Michael' | Label: NAME_GIVEN
  Entity: 'Michael' | Label: NAME_GIVEN
  Entity: 'Michael' | Label: NAME_GIVEN
    Related: 'David Carter' | Label: NAME
    Related: 'October 20th' | Label: DOB
----------------------------------------
Coreference Group ID: c74c9d15-c2e8-459a-